In [ ]:
import os 
os.chdir(r'Q:\sachuriga\Sachuriga_Python/quattrocolo-nwb4fp\src')

from neurochat.nc_data import NData
from neurochat.nc_spike import NSpike
from neurochat.nc_spatial import NSpatial
import neurochat.nc_plot as nc_plot
from neurochat.nc_lfp import NLfp
import matplotlib.pyplot as plt
import numpy as np
from pynwb import NWBHDF5IO
import matplotlib.pyplot as plt
import numpy as np
import math
import pynapple as nap
import numpy as np
from scipy import signal
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import normalize

import sys
import nwb4fp.analyses.maps as mapp
from nwb4fp.analyses.examples.tracking_plot import plot_ratemap,plot_path
from nwb4fp.analyses.fields import separate_fields_by_laplace, separate_fields_by_dilation,find_peaks,separate_fields_by_laplace_of_gaussian,calculate_field_centers,distance_to_edge_function, remove_fields_by_area, map_pass_to_unit_circle,which_field,compute_crossings
from elephant.statistics import time_histogram, instantaneous_rate
from nwb4fp.analyses import maps
from nwb4fp.analyses.data import pos2speed,speed_filtered_spikes,load_speed_fromNWB,load_units_fromNWB,get_filed_num,unit_location_ch
from scipy.ndimage import gaussian_filter
import ast
import pandas as pd
pd.set_option('display.max_rows', None)
np.set_printoptions(threshold=np.inf)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
import seaborn as sns
from itertools import chain
import pandas as pd
from nwb4fp.data.helpers import df2results, df2results_sns

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
import os

# Directory containing NWB files
folder_path = r"S:\Sachuriga/nwb/test4neo/"

# Lists to store results
results = []

# Loop through all .nwb files in the folder
for filename in os.listdir(folder_path):
    if filename.endswith('.nwb'):
        filepath = os.path.join(folder_path, filename)
        
        # Extract animal_id and session from filename
        animal_id = filename.split("neo/")[-1][:5] if "neo/" in filename else filename[:5]
        session = filename.split("_phy")[0][-1] if "_phy" in filename else "unknown"
        
        # try:
            # Load data
        npdata = nap.load_file(filepath)
        pos_cord = load_speed_fromNWB(npdata['XY_mid_brain'])

        # Process position and speed data
        raw_pos, combined_array, mask, speeds, smoothed_speed, filtered_speed = pos2speed(
            pos_cord[:,0], pos_cord[:,1], pos_cord[:,2],
            filter_speed=True, min_speed=0.05
        )
        combined_array = raw_pos
        combined_array[np.isnan(combined_array)] = 0
        smoothed_speed[np.isnan(smoothed_speed)] = 0

        # Create bins for heatmap
        x_bins = np.linspace(min(combined_array[:,1]), max(combined_array[:,1]), 51)
        y_bins = np.linspace(min(combined_array[:,2]), max(combined_array[:,2]), 51)

        # Calculate speed heatmap
        speed_heatmap = np.zeros((len(x_bins)-1, len(y_bins)-1))
        counts = np.zeros((len(x_bins)-1, len(y_bins)-1))
        counts1 = np.zeros((len(x_bins)-1, len(y_bins)-1))
        
        for i in range(len(combined_array)):
            x_idx = np.searchsorted(x_bins, combined_array[i,1]) - 1
            y_idx = np.searchsorted(y_bins, combined_array[i,2]) - 1
            if 0 <= x_idx < len(x_bins)-1 and 0 <= y_idx < len(y_bins)-1:
                speed_heatmap[x_idx, y_idx] += smoothed_speed[i]
                counts[x_idx, y_idx] += 100/len(combined_array)
                counts1[x_idx, y_idx] += 1

        # Process heatmaps
        speed_heatmap = np.divide(speed_heatmap, counts1, where=counts!=0) * 50
        speed_heatmap[counts1 == 0] = np.nan
        speed_heatmap_smoothed = gaussian_filter(speed_heatmap, sigma=2)
        H_counts_smoothed = gaussian_filter(counts, sigma=2)

        # Calculate metrics
        active_times = len(smoothed_speed[smoothed_speed>=0.05])/len(smoothed_speed)
        
        center_x_min, center_x_max = 0.25, 0.75
        center_y_min, center_y_max = 0.25, 0.75
        
        df = pd.DataFrame({'x': combined_array[:,1], 'y': combined_array[:,2]})
        df['in_center'] = (
            (df['x'] >= center_x_min) & (df['x'] <= center_x_max) & 
            (df['y'] >= center_y_min) & (df['y'] <= center_y_max)
        ).astype(int)

        time_in_center = len(np.where(df['in_center'] == 1)[0])/len(df)
        mean_speed = np.mean(smoothed_speed)*50
        
        speed_in_center = smoothed_speed[df['in_center'] == 1]
        speed_in_edge = smoothed_speed[df['in_center'] == 0]
        filter_speed_in_center = speed_in_center
        filter_speed_in_edge = speed_in_edge
        
        center_border_ratio = np.mean(filter_speed_in_center)/np.mean(filter_speed_in_edge)

        # Store results
        results.append({
            'filename': filename,
            'animal_id': animal_id,
            'session': session,
            'filter_speed_in_center': np.mean(filter_speed_in_center)*50,
            'filter_speed_in_edge': np.mean(filter_speed_in_edge)*50,
            'center_border_ratio': center_border_ratio,
            'active_times': active_times,
            'mean_speed': mean_speed,
            'smoothed_speed': smoothed_speed,
            'time_in_center': time_in_center,
            'H_counts_smoothed': H_counts_smoothed,
            'speed_heatmap_smoothed': speed_heatmap_smoothed,
            'x':df['x'],
            'y':df['y']
        })

        print(f"Processed: {filename}")
        print(f"speed in center: {np.mean(filter_speed_in_center)*50:.2f} cm/s")
        print(f"speed in border: {np.mean(filter_speed_in_edge)*50:.2f} cm/s")
        print(f"center/border speed ratio: {center_border_ratio:.2f}")
        print(f"animal is active during: {active_times*100:.2f}% of time")
        print(f"Average speed is: {mean_speed:.2f} cm/s")
        print(f"time in center: {time_in_center*100:.2f}%")
        print("-" * 50)

        # except Exception as e:
        #     print(f"Error processing {filename}: {str(e)}")
        #     continue

# Convert results to DataFrame for easy analysis
results_df = pd.DataFrame([{
    'filename': r['filename'],
    'animal_id': r['animal_id'],
    'session': r['session'],
    'filter_speed_in_center': r['filter_speed_in_center'],
    'filter_speed_in_edge': r['filter_speed_in_edge'],
    'center_border_ratio': r['center_border_ratio'],
    'active_times': r['active_times'],
    'mean_speed': r['mean_speed'],
    'smoothed_speed':r['smoothed_speed'],
    'time_in_center': r['time_in_center'],
    'H_counts_smoothed': r['H_counts_smoothed'],
    'speed_heatmap_smoothed': r['speed_heatmap_smoothed'],
    'x':r['x'],
    'y':r['y']
} for r in results])

# Save results to CSV (optional)
results_df.to_csv('S:/Sachuriga/nwb/test4neo/test/speed_analysis_results.csv', index=False)
results_df.to_pickle('S:/Sachuriga/nwb/test4neo/test/speed_analysis_results.pkl')

# You can access the heatmaps separately from the results list
# For example: results[0]['H_counts_smoothed'] for the first file's time heatmap

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Folder set
base_folder = r"Q:/sachuriga/CR_CA1_paper/Results/Locomotion"
session = "A"

# Load the data
df_loaded = pd.read_pickle('S:/Sachuriga/nwb/test4neo/test/speed_analysis_results.pkl')
df = df_loaded 

# Define control and experimental animal IDs
control_ids = ['65165', '65091', '63383', '66539', '65622']
exp_ids = ['65588', '63385', '66538', '66537', '66922']

session = ["A"]

for session in session:
    # Filter for 'A' sessions
    if session == "Total":
        df_a = df
    else:
        df_a = df[df['session'] == session]

    # Separate into control and experimental groups
    control_df = df_a[df_a['animal_id'].isin(control_ids)]
    exp_df = df_a[df_a['animal_id'].isin(exp_ids)]

    # Set a consistent figure size
    FIG_SIZE = (6, 6)  # Width, Height in inches

    # First Plot: Control KDE (Top-Left)
    g1 = sns.JointGrid(data=df2results_sns(control_df), x="x", y="y", space=0, height=FIG_SIZE[1], ratio=5)
    g1.plot_joint(sns.kdeplot, fill=True, clip=((0, 1), (0, 1)), thresh=0, levels=100, cmap="Greys")
    g1.plot_marginals(sns.histplot, color=sns.color_palette("Greys")[-1], alpha=0.8, edgecolor="none")
    g1.ax_joint.set_xticks([0, 1])
    g1.ax_joint.set_xticklabels([0, 50])
    g1.ax_joint.set_yticks([0, 1])
    g1.ax_joint.set_yticklabels([50, 0])
    g1.ax_joint.set_xlabel('X Distance (cm)')
    g1.ax_joint.set_ylabel('Y Distance (cm)')
    g1.ax_joint.set_title('')  # Clear joint title to avoid overlap
    g1.figure.suptitle('Control KDE', y=1.05)  # Add title above the figure
    # Remove axis spines
    for spine in g1.ax_joint.spines.values():
        spine.set_visible(False)
    for spine in g1.ax_marg_x.spines.values():
        spine.set_visible(False)
    for spine in g1.ax_marg_y.spines.values():
        spine.set_visible(False)
    g1.figure.set_size_inches(FIG_SIZE)
    g1.figure.savefig(fr'{base_folder}/Session_{session}_control_activity_kde.pdf', format='pdf', bbox_inches='tight')
    g1.figure.savefig(fr'{base_folder}/Session_{session}_control_activity_kde.png', format='png', bbox_inches='tight', dpi=300)

    # Second Plot: Experimental Heatmap (Top-Right)
    fig2 = plt.figure(figsize=FIG_SIZE)
    ax2 = fig2.add_subplot(111)
    data_speedss_exp = df2results(exp_df)
    sns.heatmap(data_speedss_exp, cmap="Greys", annot=False, linewidths=0, vmin=0, ax=ax2, square=True,
                xticklabels=np.linspace(0, 50, 6), yticklabels=np.linspace(50, 0, 6),
                cbar_kws={'shrink': 0.5, 'aspect': 10, 'pad': 0.02})  # Adjusted colorbar padding
    n_rows, n_cols = data_speedss_exp.shape
    ax2.set_xticks([0, n_cols - 1])
    ax2.set_xticklabels([0, 50])
    ax2.set_yticks([0, n_rows - 1])
    ax2.set_yticklabels([50, 0])
    ax2.set_xlabel('X Distance (cm)')
    ax2.set_ylabel('Y Distance (cm)')
    ax2.set_title('Experimental Heatmap')  # Keep title on heatmap
    # Remove axis spines
    for spine in ax2.spines.values():
        spine.set_visible(False)
    fig2.subplots_adjust(left=0.15, right=0.85, top=0.85, bottom=0.15)  # Consistent margins
    fig2.savefig(fr'{base_folder}/Session_{session}_exp_heatmap.pdf', format='pdf', bbox_inches='tight')
    fig2.savefig(fr'{base_folder}/Session_{session}_exp_heatmap.png', format='png', bbox_inches='tight', dpi=300)

    # Third Plot: Experimental KDE (Bottom-Left)
    g3 = sns.JointGrid(data=df2results_sns(exp_df), x="x", y="y", space=0, height=FIG_SIZE[1], ratio=5)
    g3.plot_joint(sns.kdeplot, fill=True, clip=((0, 1), (0, 1)), thresh=0, levels=100, cmap="Greys")
    g3.plot_marginals(sns.histplot, color=sns.color_palette("Greys")[-1], alpha=0.8, edgecolor="none")
    g3.ax_joint.set_xticks([0, 1])
    g3.ax_joint.set_xticklabels([0, 50])
    g3.ax_joint.set_yticks([0, 1])
    g3.ax_joint.set_yticklabels([50, 0])
    g3.ax_joint.set_xlabel('X Distance (cm)')
    g3.ax_joint.set_ylabel('Y Distance (cm)')
    g3.ax_joint.set_title('')  # Clear joint title to avoid overlap
    g3.figure.suptitle(f'Session {session} Experimental KDE', y=1.05)  # Add title above the figure
    # Remove axis spines
    for spine in g3.ax_joint.spines.values():
        spine.set_visible(False)
    for spine in g3.ax_marg_x.spines.values():
        spine.set_visible(False)
    for spine in g3.ax_marg_y.spines.values():
        spine.set_visible(False)
    g3.figure.set_size_inches(FIG_SIZE)
    g3.figure.savefig(fr'{base_folder}/Session_{session}_exp_activity_kde.pdf', format='pdf', bbox_inches='tight')
    g3.figure.savefig(fr'{base_folder}/Session_{session}_exp_activity_kde.png', format='png', bbox_inches='tight', dpi=300)

    # Fourth Plot: Control Heatmap (Bottom-Right)
    fig4 = plt.figure(figsize=FIG_SIZE)
    ax4 = fig4.add_subplot(111)
    data_speedss_control = df2results(control_df)
    sns.heatmap(data_speedss_control, cmap="Greys", annot=False, linewidths=0, vmin=0, ax=ax4, square=True,
                xticklabels=np.linspace(0, 50, 6), yticklabels=np.linspace(50, 0, 6),
                cbar_kws={'shrink': 0.5, 'aspect': 10, 'pad': 0.02})  # Adjusted colorbar padding
    n_rows, n_cols = data_speedss_control.shape
    ax4.set_xticks([0, n_cols - 1])
    ax4.set_xticklabels([0, 50])
    ax4.set_yticks([0, n_rows - 1])
    ax4.set_yticklabels([50, 0])
    ax4.set_xlabel('X Distance (cm)')
    ax4.set_ylabel('Y Distance (cm)')
    ax4.set_title('Control Heatmap')  # Keep title on heatmap
    # Remove axis spines
    for spine in ax4.spines.values():
        spine.set_visible(False)
    fig4.subplots_adjust(left=0.15, right=0.85, top=0.85, bottom=0.15)  # Consistent margins
    fig4.savefig(fr'{base_folder}/Session_{session}_control_heatmap.pdf', format='pdf', bbox_inches='tight')
    fig4.savefig(fr'{base_folder}/Session_{session}_control_heatmap.png', format='png', bbox_inches='tight', dpi=300)

    # Display all plots
    plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import pandas as pd

base_folder = r"Q:/sachuriga/CR_CA1_paper/Results/Locomotion"

# Load the data
df_loaded = pd.read_pickle('S:/Sachuriga/nwb/test4neo/test/speed_analysis_results.pkl')
df = df_loaded 

# Define control and experimental animal IDs
control_ids = ['65165', '65091', '63383', '66539', '65622']
exp_ids = ['65588', '63385', '66538', '66537', '66922']

session = ["A","B","C","Total"]

for session in session:
    # Filter for 'A' sessions
    if session == "Total":
        df_a = df
    else:
        df_a = df[df['session'] == session]

    # Separate into control and experimental groups
    control_df = df_a[df_a['animal_id'].isin(control_ids)]
    exp_df = df_a[df_a['animal_id'].isin(exp_ids)]

    # Set Seaborn theme
    sns.set_theme(style="ticks")

    # Statistical comparisons for scalar metrics
    metrics = ['filter_speed_in_center', 'filter_speed_in_edge', 'center_border_ratio', 
            'active_times', 'mean_speed', 'time_in_center']

    # Create figure with 3x2 subplots
    fig, axes = plt.subplots(3, 2, figsize=(6, 14))
    axes = axes.flatten()  # Flatten the 2D array of axes for easier iteration

    # Define custom colors
    control_color = sns.color_palette("Greys")[-1]  # Dark blue for Control
    exp_color = sns.color_palette("Greys")[0]       # Light blue for Experimental

    for idx, metric in enumerate(metrics):
        control_values = control_df[metric].dropna()
        exp_values = exp_df[metric].dropna()
        
        if len(control_values) > 0 and len(exp_values) > 0:
            control_mean = control_values.mean()
            exp_mean = exp_values.mean()
            control_sem = control_values.sem()
            exp_sem = exp_values.sem()
            
            print(f"\nComparison for {metric}:")
            print(f"Control mean: {control_mean:.2f} ± {control_sem:.2f}")
            print(f"Experimental mean: {exp_mean:.2f} ± {exp_sem:.2f}")
            
            # Mann-Whitney U test
            u_stat, p_val = stats.mannwhitneyu(control_values, exp_values, alternative='two-sided')
            print(f"Mann-Whitney U statistic: {u_stat:.2f}, p-value: {p_val:.4f}")
            
            # Prepare data for Seaborn plotting
            plot_df = pd.DataFrame({
                'value': pd.concat([control_values, exp_values]),
                'group': ['Control'] * len(control_values) + ['Experimental'] * len(exp_values)
            })
            
            # Create vertical boxplot on the specific subplot
            boxplot = sns.boxplot(
                data=plot_df,
                x='group',
                y='value',
                ax=axes[idx],
                palette={"Control": control_color, "Experimental": exp_color},
                whis=[0, 100],
                width=.6
            )

            # Set the alpha (transparency) for the boxplot components
            for patch in boxplot.patches:
                patch.set_alpha(0.8)  # Set the alpha value here
            
            # Add individual points with matching colors
            
            # Add individual points with matching colors
            sns.stripplot(
                data=plot_df,
                x='group',
                y='value',
                ax=axes[idx],
                size=4,
                hue='group',  # Use hue to assign colors per group
                #palette={"Control": control_color, "Experimental": exp_color},  # Same colors as boxplot
                palette={"Control": "black", "Experimental": "black"},  # Same colors as boxplot
                alpha=1,
                legend=False  # Remove legend from stripplot
            )
            
            # Set title and labels
            axes[idx].set_title(f'{metric} Comparison')
            axes[idx].set_ylabel(metric)
            axes[idx].set_xlabel('Group')
            axes[idx].yaxis.grid(False)
            axes[idx].set(xlabel="")
            
            # Set y-axis to start from 0
            axes[idx].set_ylim(bottom=0, top=axes[idx].get_ylim()[1])  # Fixed 'custom' to 'bottom'
            
            # Add p-value at the top of the plot
            axes[idx].text(0.5, 0.2, f'p = {p_val:.4f}', 
                        horizontalalignment='center', 
                        verticalalignment='top', 
                        transform=axes[idx].transAxes, 
                        fontsize=10)
            
            # Remove top and right spines, keep bottom (x) and left (y) axes
            axes[idx].spines['top'].set_visible(False)
            axes[idx].spines['right'].set_visible(False)
            axes[idx].spines['bottom'].set_visible(True)  # Keep x-axis
            axes[idx].spines['left'].set_visible(True)    # Keep y-axis

    fig.savefig(fr'{base_folder}/{session}_sample4session_metrics_comparison.eps', format='eps', bbox_inches='tight')
    fig.savefig(fr'{base_folder}/{session}_sample4session_metrics_comparison.png', format='png', bbox_inches='tight')
    # Adjust layout to prevent overlap
    plt.tight_layout()
    plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import pandas as pd

# Set Seaborn theme
sns.set_theme(style="ticks")

# Define control and experimental animal IDs
control_ids = ['65165', '65091', '63383', '66539', '65622']
exp_ids = ['65588', '63385', '66538', '66537', '66922']

base_folder = r"Q:/sachuriga/CR_CA1_paper/Results/Locomotion"

# Load the data
df_loaded = pd.read_pickle('S:/Sachuriga/nwb/test4neo/test/speed_analysis_results.pkl')
df = df_loaded 

# Define control and experimental animal IDs
control_ids = ['65165', '65091', '63383', '66539', '65622']
exp_ids = ['65588', '63385', '66538', '66537', '66922']

session = ["A","B","C","Total"]

for session in session:
    
        # Filter for 'A' sessions
    if session == "Total":
        df_a = df
    else:
        df_a = df[df['session'] == session]

    # Separate into control and experimental groups
    control_df = df_a[df_a['animal_id'].isin(control_ids)]
    exp_df = df_a[df_a['animal_id'].isin(exp_ids)]
        
    # Assuming control_df and exp_df are already defined from previous context
    # Filter data for specific animal IDs and calculate mean per animal
    control_animals = control_df[control_df['animal_id'].isin(control_ids)].groupby('animal_id').mean(numeric_only=True)
    exp_animals = exp_df[exp_df['animal_id'].isin(exp_ids)].groupby('animal_id').mean(numeric_only=True)

    # Statistical comparisons for scalar metrics
    metrics = ['filter_speed_in_center', 'filter_speed_in_edge', 'center_border_ratio', 
            'active_times', 'mean_speed', 'time_in_center']

    # Create figure with 3x2 subplots
    fig, axes = plt.subplots(3, 2, figsize=(6, 14))
    axes = axes.flatten()  # Flatten the 2D array of axes for easier iteration

    # Define custom colors
    control_color = sns.color_palette("Greys")[-1]  # Dark blue for Control
    exp_color = sns.color_palette("Greys")[0]       # Light blue for Experimental


    for idx, metric in enumerate(metrics):
        # Get per-animal averages
        control_values = control_animals[metric].dropna()
        exp_values = exp_animals[metric].dropna()
        
        if len(control_values) > 0 and len(exp_values) > 0:
            control_mean = control_values.mean()
            exp_mean = exp_values.mean()
            control_sem = control_values.sem()
            exp_sem = exp_values.sem()
            
            # Sample sizes (number of animals)
            control_n = len(control_values)
            exp_n = len(exp_values)
            
            print(f"\nComparison for {metric}:")
            print(f"Control mean (n={control_n}): {control_mean:.2f} ± {control_sem:.2f}")
            print(f"Experimental mean (n={exp_n}): {exp_mean:.2f} ± {exp_sem:.2f}")
            
            # Mann-Whitney U test
            u_stat, p_val = stats.mannwhitneyu(control_values, exp_values, alternative='two-sided')
            print(f"Mann-Whitney U statistic: {u_stat:.2f}, p-value: {p_val:.4f}")
            
            # Prepare data for Seaborn plotting
            plot_df = pd.DataFrame({
                'value': pd.concat([control_values, exp_values]),
                'group': ['Control'] * len(control_values) + ['Experimental'] * len(exp_values)
            })
            
            # Create the boxplot without alpha
            boxplot = sns.boxplot(
                data=plot_df,
                x='group',
                y='value',
                ax=axes[idx],
                palette={"Control": control_color, "Experimental": exp_color},
                whis=[0, 100],
                width=.6
            )

            # Set the alpha (transparency) for the boxplot components
            for patch in boxplot.patches:
                patch.set_alpha(0.8)  # Set the alpha value here
            
            # Add individual points with matching colors
            sns.stripplot(
                data=plot_df,
                x='group',
                y='value',
                ax=axes[idx],
                size=4,
                hue='group',  # Use hue to assign colors per group
                #palette={"Control": control_color, "Experimental": exp_color},  # Same colors as boxplot
                palette={"Control": "black", "Experimental": "black"},  # Same colors as boxplot
                alpha=1,
                jitter=0.1,
                legend=False  # Remove legend from stripplot
            )
            
            # Set title and labels
            axes[idx].set_title(f'{metric} Comparison')
            axes[idx].set_ylabel(metric)
            axes[idx].set_xlabel('Group')
            axes[idx].yaxis.grid(False)
            axes[idx].set(xlabel="")
            
            # Set y-axis to start from 0
            axes[idx].set_ylim(bottom=0, top=axes[idx].get_ylim()[1])  # Fixed 'custom' to 'bottom'
            
            # Add p-value and sample sizes at the top of the plot
            axes[idx].text(0.5, 0.2, f'p = {p_val:.4f}\nControl n={control_n}, Exp n={exp_n}', 
                        horizontalalignment='center', 
                        verticalalignment='top', 
                        transform=axes[idx].transAxes, 
                        fontsize=10)
            
            # Remove top and right spines, keep bottom (x) and left (y) axes
            axes[idx].spines['top'].set_visible(False)
            axes[idx].spines['right'].set_visible(False)
            axes[idx].spines['bottom'].set_visible(True)  # Keep x-axis
            axes[idx].spines['left'].set_visible(True)    # Keep y-axis

    fig.savefig(fr'{base_folder}/{session}_sample4animal_metrics_comparison.eps', format='eps', bbox_inches='tight')
    fig.savefig(fr'{base_folder}/{session}_sample4animal_metrics_comparison.png', format='png', bbox_inches='tight')
    # Adjust layout to prevent overlap
    plt.tight_layout()
    plt.show()